In [ ]:
import os
import numpy as np
import pandas as pd

OUTLINE_ROOT = "all_outlines_cleaned_434"

outlines = []
countries = []
leaf_ids = []

for fname in sorted(os.listdir(OUTLINE_ROOT)):
    if not fname.endswith(".npy"):
        continue
    
    path = os.path.join(OUTLINE_ROOT, fname)
    contour = np.load(path)
    
    # Extract country from filename
    country = fname.split("_")[0]
    
    outlines.append(contour)
    countries.append(country)
    leaf_ids.append(fname)

print("Total outlines loaded:", len(outlines))
print("Example contour shape:", outlines[78].shape)
print("Countries found:", set(countries))

In [ ]:
from pyefd import elliptic_fourier_descriptors
import numpy as np
import pandas as pd

harmonics = 20

efd_features = []

for contour in outlines:
    
    contour = np.array(contour, dtype=np.float64)
    
    coeffs = elliptic_fourier_descriptors(
        contour,
        order=harmonics,
        normalize=True
    )
    
    efd_features.append(coeffs.flatten())

efd_df = pd.DataFrame(efd_features)

print("EFD shape:", efd_df.shape)

In [ ]:
#remove first harmonic
efd_df_clean = efd_df.drop(columns=[0,1,2,3])

print("After removing first harmonic:", efd_df_clean.shape)
efd_df_clean["country"] = countries
efd_df_clean = efd_df_clean.reset_index(drop=True)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

# Separate features
X = efd_df_clean.drop(columns=["country"])

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca_efd = PCA()
X_pca = pca_efd.fit_transform(X_scaled)

# Put into dataframe
efd_pca_df = pd.DataFrame(X_pca[:, :5], columns=["PC1","PC2","PC3","PC4","PC5"])
efd_pca_df["country"] = efd_df_clean["country"].values

print("PCA completed")

In [ ]:
#variance table
explained = pca_efd.explained_variance_ratio_ * 100
cumulative = np.cumsum(explained)

variance_table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained))],
    "Variance (%)": explained,
    "Cumulative (%)": cumulative
}).round (2)

variance_table.head(10).to_csv('efd_pca_variance.csv', index = True)

In [ ]:
#shape reconstruction on PCA

from pyefd import reconstruct_contour
import numpy as np
import matplotlib.pyplot as plt

harmonics = 20

def reconstruct_shape(pc_index=2, sd_multiplier=0):
    
    # Start at origin in PCA space
    pc_vector = np.zeros(pca_efd.n_components_)
    
    # Get SD of that PC
    pc_std = np.std(X_pca[:, pc_index])
    
    # Move along axis
    pc_vector[pc_index] = sd_multiplier * pc_std
    
    # Inverse PCA transform
    coeffs_scaled = pca_efd.inverse_transform(pc_vector)
    
    # Reverse scaling
    coeffs = scaler.inverse_transform([coeffs_scaled])[0]
    
    # Add back first harmonic zeros
    coeffs_full = np.concatenate([np.zeros(4), coeffs])
    
    coeffs_full = coeffs_full.reshape((harmonics, 4))
    
    contour = reconstruct_contour(coeffs_full, locus=(0,0), num_points=400)
    
    return contour


mean_shape = reconstruct_shape(pc_index=2, sd_multiplier=0)
plus2 = reconstruct_shape(pc_index=2, sd_multiplier=2)
minus2 = reconstruct_shape(pc_index=2, sd_multiplier=-2)


plt.figure(figsize=(6,6))
plt.plot(mean_shape[:,0], mean_shape[:,1], label="Mean")
plt.plot(plus2[:,0], plus2[:,1], linestyle="--", label="+2 SD PC3")
plt.plot(minus2[:,0], minus2[:,1], linestyle=":", label="-2 SD PC3")

plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.title("Outline Deformation Along PC3 (EFD)")
plt.show()

In [ ]:
#permanova on EFD space
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import permanova
import pandas as pd

# Create string IDs
ids = efd_df_clean.index.astype(str)

# Create distance matrix
distance_matrix = DistanceMatrix(
    squareform(pdist(X_scaled)),
    ids=ids
)

# Create grouping dataframe with matching index
grouping_df = pd.DataFrame({
    "country": efd_df_clean["country"].values
}, index=ids)

# Run PERMANOVA
permanova_result = permanova(
    distance_matrix,
    grouping_df,
    column="country",
    permutations=999
)

print(permanova_result)
efd_centroids = efd_pca_df.groupby("country")[["PC1","PC2","PC3"]].mean()
efd_centroids

In [ ]:
#efd pca centroids distance matrix
from scipy.spatial.distance import pdist, squareform
import pandas as pd

efd_distances = pd.DataFrame(
    squareform(pdist(efd_centroids)),
    index=efd_centroids.index,
    columns=efd_centroids.index
)

efd_distances